# ⚾ Homework: Moneyball Part 2, Runs Allowed
In class we found that **on-base percentage** and **slugging percentage** drive **runs scored**. But the A's win games through **run differential** (runs scored − runs allowed), so preventing runs matters just as much.

In this homework you'll build a regression model for **runs allowed**.

Run the setup cells below first. They load the same data and model-report helper we used in class.

In [ ]:
#THIS IS MY OWN BASH THING FOR VIRTUALIZATION OF PYTHON ENVIRONMENT. CODESPACES RAN OUT FOR ME SO IM DOING THIS: 
cd ~/Desktop/"CTP Folder"/"CTP WEEK 08"/ds-fall-2026
source .venv/bin/activate
jupyter notebook


## 🧰 Setup: load our tools

In [ ]:
import pandas as pd
from pathlib import Path

# statsmodels builds our regression models
import statsmodels.api as sm
from statsmodels.tools.eval_measures import rmse

# sklearn scores our models and splits data into training and testing sets
from sklearn import metrics
from sklearn.model_selection import train_test_split

: 

## 🔎 Load the data and travel back to 2001
Every row is **one team in one season** (for example, the 2001 Oakland A's).

Look at what the code below does:
1. Reads `baseball.csv`.
2. Renames the columns: abbreviations are spelled out using a dictionary, and every label is made lower case with underscores for spaces (`RS` becomes `runs_scored`).
3. Keeps only seasons **before 2002**. This is our "time machine": we pretend we're DePodesta in the 2001–02 off-season.
4. Creates **run differential**: `runs_scored − runs_allowed`. It's stored as `runs_diff`.

In [ ]:
# Works whether this notebook sits in the week folder or inside exercise/
data_path = next(p for p in ['data/baseball.csv', '../data/baseball.csv'] if Path(p).exists())
df = pd.read_csv(data_path)

In [ ]:
# Spell out the abbreviated column names so the data is easier to read
column_names = {
    'RS': 'runs_scored',
    'RA': 'runs_allowed',
    'W': 'wins',
    'OBP': 'on_base_percentage',
    'SLG': 'slugging_percentage',
    'BA': 'batting_average',
    'G': 'games',
    'OOBP': 'opponent_obp',
    'OSLG': 'opponent_slg',
    'RankSeason': 'rank_season',
    'RankPlayoffs': 'rank_playoffs',
}
df = df.rename(columns=column_names)

# Make every column label lower case (e.g. Team -> team, Year -> year)
df.columns = df.columns.str.lower()

In [ ]:
# Time machine: keep only seasons before Moneyball (2002)
df = df[df['year'] < 2002].reset_index(drop=True)

In [ ]:
# Run differential: runs scored minus runs allowed
df['runs_diff'] = df['runs_scored'] - df['runs_allowed']

df.head()

#### Data Dictionary

| **Column**              | **Original** | **Description**                                          |
| ----------------------- | ------------ | -------------------------------------------------------- |
| **team**                | Team         | Name of the team                                         |
| **league**              | League       | Name of the league                                       |
| **year**                | Year         | Year of the season                                       |
| **runs_scored**         | RS           | Runs scored by the team                                  |
| **runs_allowed**        | RA           | Runs allowed by the team                                 |
| **wins**                | W            | Number of wins                                           |
| **on_base_percentage**  | OBP          | On-Base Percentage                                       |
| **slugging_percentage** | SLG          | Slugging Percentage                                      |
| **batting_average**     | BA           | Batting Average                                          |
| **playoffs**            | Playoffs     | Whether the team made the playoffs (`1 = Yes`, `0 = No`) |
| **rank_season**         | RankSeason   | Regular-season rank among playoff teams (blank if missed playoffs) |
| **rank_playoffs**       | RankPlayoffs | Playoff finish, 1 = won the World Series (blank if missed playoffs) |
| **games**               | G            | Number of games played                                   |
| **opponent_obp**        | OOBP         | Opponent On-Base Percentage                              |
| **opponent_slg**        | OSLG         | Opponent Slugging Percentage                             |
| **runs_diff**           | (new)        | Runs scored minus runs allowed (`runs_scored - runs_allowed`) |


> 💡 **Baseball stats in plain English** (you don't need to be a fan!)
> * **BA (Batting Average)**: hits ÷ at-bats. *How often does a batter get a hit?* This is the stat **traditional scouts loved**.
> * **OBP (On-Base Percentage)**: how often a batter **reaches base by any means** (hit, walk, hit-by-pitch). A walk counts here but not in BA.
> * **SLG (Slugging Percentage)**: total bases ÷ at-bats. Measures **power**: a home run is worth 4, a single is worth 1.
> * **OOBP / OSLG**: the same stats, but for the **opponents' hitters** against our pitchers. Lower is better for us.
>
> 🎯 **The Moneyball bet:** the market paid big money for high **batting average**, but walks (which boost OBP) were nearly free. If OBP matters more than BA for scoring runs, the A's can buy runs cheaply.

## 🛠️ Helper function
**`print_model_report`** is the same helper we used in class. It prints the coefficients and p-values, then R², RMSE, and the average of the dependent variable. By default it scores the model on the data it was trained on. Pass in different `X` and `y` (like a test set) to score it on that data instead.

In [ ]:
def print_model_report(model, X=None, y=None):
    # Coefficient and p-value for each variable
    report = pd.DataFrame({'coefficient': model.params, 'p_value': model.pvalues})
    print(report.round(4))
    print()

    # By default, score the model on the data it was trained on
    # (model.model.exog holds the X values; model.model.endog holds the actual y values)
    if X is None:
        X = model.model.exog
        y = model.model.endog

    # How well the model fits
    y_pred = model.predict(X)
    print('R-squared:', round(metrics.r2_score(y, y_pred), 4))
    print('RMSE:', round(rmse(y, y_pred), 2))
    print(f'Average {model.model.endog_names}:', round(y.mean(), 2))

---
## 🧤 Your task: How do we ALLOW fewer runs?
Winning is about **run differential**, so preventing runs counts just as much as scoring them. Let's flip the question around and use the **opponents'** hitting stats against our pitchers:

* **Dependent variable:** `runs_allowed`
* **Try predictors:** `opponent_obp` (opponents' on-base %) and `opponent_slg` (opponents' slugging %)

⚠️ Check for nulls first (`df.info()`): `opponent_obp` and `opponent_slg` are **missing for most of the older seasons**. Drop those rows first (e.g. `df.dropna(subset=['opponent_obp', 'opponent_slg'])`), or the model will error out.

**Your turn:** build and evaluate this model by following the steps below. Then decide: do the coefficients have the sign you'd expect? How does the R² compare with the runs-scored model from class (R² ≈ 0.93 using `on_base_percentage` and `slugging_percentage`)?

### 1. Check for nulls and drop missing rows

In [ ]:
# Check for nulls, then drop the rows that are missing opponent_obp or opponent_slg
df.info()

# Count the number of missing values in the two columns we want to use
missingOpponentObpCount = df['opponent_obp'].isna().sum()
missingOpponentSlgCount = df['opponent_slg'].isna().sum()
print('Missing opponent_obp:', missingOpponentObpCount)
print('Missing opponent_slg:', missingOpponentSlgCount)

# dropna(subset=[...]) throws away any row that is blank in EITHER of those two columns.
df_with_opponent_stats = df.dropna(subset=['opponent_obp', 'opponent_slg'])

print('Rows before dropping:', len(df))
print('Rows after dropping: ', len(df_with_opponent_stats))

print('Seasons left:', sorted(df_with_opponent_stats['year'].unique()))


### 2. Define X and y
Add a constant to `X` (`sm.add_constant`) so the model can learn an intercept.

In [ ]:
# Define X (the predictors, plus a constant) and y (runs_allowed)

# The columns the model is allowed to see
predictor_columns = ['opponent_obp', 'opponent_slg']

# The column the model tries to guess
target_column = 'runs_allowed'

# sm.add_constant tacks on a column of all 1s named "const".
# That column lets the model learn an intercept
X = sm.add_constant(df_with_opponent_stats[predictor_columns])

# y is just the single column we are trying to predict
y = df_with_opponent_stats[target_column]

X.head()

### 3. Build the model on all the data
Fit the model with `sm.OLS(y, X).fit()`, then print the results with `print_model_report`.

In [ ]:
# Build the model on all the data and print the model report

# sm.OLS(y, X) sets up "Ordinary Least Squares" regression. Also y comes first in statsmodels.
# .fit() then finds the coefficients that make the predictions miss by the smallest total amount.
runs_allowed_model = sm.OLS(y, X).fit()

# Our helper prints each coefficient and p-value, then R-squared, RMSE, and the average runs_allowed.
# With no X or y passed in, it scores the model on the same data it was trained on.
print_model_report(runs_allowed_model)

### 4. Evaluate on a train/test split
Split the data with `train_test_split(X, y, test_size=0.2, random_state=42)` so everyone gets the same numbers. Fit a model on the training data, then score it on the testing data with `print_model_report(model, X_test, y_test)`.

In [ ]:
# Split the data, build the model on the training data, and score it on the testing data

# test_size=0.2 so 20% of the rows are locked away for testing.
# random_state=42 so the shuffle is the same every run, so everyone gets the same numbers.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

print('Training rows:', len(X_train))
print('Testing rows: ', len(X_test))

#brand-new model that ONLY ever sees the training rows
runs_allowed_model_trained_on_split = sm.OLS(y_train, X_train).fit()

# Score that model on the rows it hasnt seen before
# The coefficients printed are from the training fit and the R-squared and RMSE are on the test set
print_model_report(runs_allowed_model_trained_on_split, X_test, y_test)

### 5. Interpret your results
✍️ **Answer here:** Do the coefficients have the sign you'd expect? How does the R² compare with the runs-scored model? How does the testing score compare with the score on all the data?


1. Do the coefficients have the sign I'd expect?
Yeah, both are positive, which makes sense. If the other team's hitters are getting on base more and hitting for more power against our pitchers, we're going to give up more runs. Both stats also came out as clearly important, so it's not just random noise. The OBP number is bigger than the SLG one, so just like with scoring runs in class, getting on base matters more than power. And nothing weird happened this time, like the batting average thing going negative in class.

2. How does the R² compare with the runs-scored model?
It's a little lower, but still really good. This model got about 0.91 compared to about 0.93 for the runs-scored one. One reason it might be lower is that we only had 90 team-seasons to work with (1999 to 2001), since the opponent stats are missing for the older years. Opponent stats also depend on things like defense, luck, and the ballpark, so there's a little more noise.

3. How does the testing score compare with the score on all the data?
It's worse on the test data, about 0.85 compared to 0.91. That's normal, since the model did better on data it already trained on. It's like doing great on a practice test you already saw the answers to, then doing a little worse on a brand-new one. It's still a solid score, so the model holds up pretty well. The test set is pretty small though (only 18 rows), so the number could change with a different split.